# RefMet PubMed/PubChem Fallback Reranker Evaluation

This notebook evaluates whether a lightweight lexical/chemical reranker can improve the saved PubMed/PubChem fallback candidates from `refmet_harmonization_publication_pipeline.ipynb`.

It does **not** re-embed strings and does **not** rerun FAISS. It reuses the saved files in the source run's `artifacts/` folder, especially:

- `pubmed_fallback_neighbor_scores.npy`
- `pubmed_fallback_neighbor_indices.npy`
- `pubmed_fallback_surfaces.csv`
- `refmet_target_dictionary.csv`
- `refmet_predictions_with_decisions.csv`

The main question is: when RefMet retrieval missed the gold RefMet target in the top 10 and the query was routed to PubMed/PubChem fallback, did the saved PubMed top-256 candidates contain the same PubChem CID as the gold RefMet target, and can reranking move it higher?

## 1. Mount Google Drive

Run this cell in Colab. If you are running locally, skip it.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', repr(exc))

## 2. Configure The Reuse-Only Reranker Run

This is a CPU-only evaluation. It can run on a GPU runtime, but it should not use GPU memory because all embeddings and FAISS searches are already done.

For a full publication run, keep `POOL_K = 256`. Lower it only for a quick smoke test.

In [ ]:
import os
from pathlib import Path

# Update this if your repo lives somewhere else in Drive.
PROJECT_DIR = Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer')
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd().resolve()

SOURCE_RUN_NAME = 'refmet_harmonization_publication_pipeline'
OUTPUT_NAME = 'pubmed_fallback_reranker_evaluation'

# Reuse the full saved PubMed/PubChem fallback candidate pool.
POOL_K = 256
TOP_X = 10

# Tuning is only on validation cases; final metrics are held-out test cases.
RANDOM_SEARCH_TRIALS = 120
TUNING_MAX_QUERIES = 10_000
RANDOM_SEED = 7

# Larger chunks are faster but use more RAM. 1024 is conservative; 2048 is usually fine on high-RAM Colab.
CHUNK_SIZE = 1024

# Copying the .npy arrays and large surface CSV from Drive to local scratch usually makes reads faster.
COPY_INPUTS_TO_SCRATCH = True
FORCE = False

env_values = {
    'CHEMSYN_PROJECT_DIR': str(PROJECT_DIR),
    'CHEMSYN_SOURCE_RUN_NAME': SOURCE_RUN_NAME,
    'CHEMSYN_OUTPUT_NAME': OUTPUT_NAME,
    'CHEMSYN_POOL_K': str(POOL_K),
    'CHEMSYN_TOP_X': str(TOP_X),
    'CHEMSYN_RANDOM_SEARCH_TRIALS': str(RANDOM_SEARCH_TRIALS),
    'CHEMSYN_TUNING_MAX_QUERIES': str(TUNING_MAX_QUERIES),
    'CHEMSYN_RANDOM_SEED': str(RANDOM_SEED),
    'CHEMSYN_CHUNK_SIZE': str(CHUNK_SIZE),
    'CHEMSYN_COPY_INPUTS_TO_SCRATCH': str(COPY_INPUTS_TO_SCRATCH),
    'CHEMSYN_FORCE': str(FORCE),
}
for key, value in env_values.items():
    os.environ[key] = value

print('Project:', PROJECT_DIR)
print('Source results:', PROJECT_DIR / 'results' / SOURCE_RUN_NAME)
print('Output:', PROJECT_DIR / 'results' / SOURCE_RUN_NAME / OUTPUT_NAME)
print('Pool K:', POOL_K)
print('Random search trials:', RANDOM_SEARCH_TRIALS)
print('Tuning max validation queries:', TUNING_MAX_QUERIES)
print('Copy inputs to scratch:', COPY_INPUTS_TO_SCRATCH)

## 3. Verify Required Inputs

This cell checks for the saved artifacts from the final publication pipeline. If a file is missing, do not re-embed; copy the missing artifact from Drive/results first.

In [ ]:
source_results = PROJECT_DIR / 'results' / SOURCE_RUN_NAME
artifact_dir = source_results / 'artifacts'
required = [
    source_results / 'refmet_predictions_with_decisions.csv',
    artifact_dir / 'refmet_target_dictionary.csv',
    artifact_dir / 'pubmed_fallback_surfaces.csv',
    artifact_dir / 'pubmed_fallback_neighbor_scores.npy',
    artifact_dir / 'pubmed_fallback_neighbor_indices.npy',
]

for path in required:
    parts = path.with_suffix(path.suffix + '.parts')
    status = 'OK' if path.exists() else ('PARTS' if parts.exists() else 'MISSING')
    size = path.stat().st_size / (1024**3) if path.exists() else None
    print(f'{status:7} {path}' + (f' ({size:.2f} GB)' if size is not None else ''))

missing = [p for p in required if not p.exists() and not p.with_suffix(p.suffix + '.parts').exists()]
if missing:
    raise FileNotFoundError('Missing required saved artifacts: ' + ', '.join(str(p) for p in missing))

## 4. Run The PubMed/PubChem Fallback Reranker Evaluation

Expected work:

1. Load saved RefMet predictions and target CIDs.
2. Load saved PubMed/PubChem neighbor scores and indices.
3. Load the saved PubMed/PubChem fallback surface table.
4. Tune reranker weights on validation fallback cases.
5. Evaluate same-CID recovery on held-out test fallback cases.
6. Save metrics, top-10 reranked candidates, examples, and plots.

No transformer model should load in this step.

In [ ]:
import subprocess
import sys

candidates = [
    PROJECT_DIR / 'scripts' / 'run_pubmed_fallback_reranker_evaluation.py',
    Path.cwd() / 'scripts' / 'run_pubmed_fallback_reranker_evaluation.py',
    Path.cwd().parent / 'scripts' / 'run_pubmed_fallback_reranker_evaluation.py',
]
script_path = next((p for p in candidates if p.exists()), None)
if script_path is None:
    raise FileNotFoundError('Could not find scripts/run_pubmed_fallback_reranker_evaluation.py')

print('Running:', script_path)
cmd = [sys.executable, str(script_path), '--project-dir', str(PROJECT_DIR)]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
return_code = proc.wait()
if return_code != 0:
    raise RuntimeError(f'PubMed fallback reranker evaluation failed with exit code {return_code}')

## 5. Inspect The Results

The most important file is `pubmed_fallback_reranker_metrics.csv`. If `Hit@256` is low, the saved PubMed fallback search did not retrieve the true same-CID candidate often enough. If `Hit@256` is high but `Hit@10` is low, reranking is the bottleneck.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown

output_dir = PROJECT_DIR / 'results' / SOURCE_RUN_NAME / OUTPUT_NAME
metrics_path = output_dir / 'pubmed_fallback_reranker_metrics.csv'
summary_path = output_dir / 'pubmed_fallback_reranker_summary.md'
input_summary_path = output_dir / 'input_subset_summary.json'

print('Output directory:', output_dir)
if input_summary_path.exists():
    print(json.dumps(json.loads(input_summary_path.read_text()), indent=2))
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics)
if summary_path.exists():
    display(Markdown(summary_path.read_text()))

## 6. View Example Cases

Use these examples for human review. They are grouped by whether the tuned PubMed reranker recovered the gold same-CID candidate.

In [ ]:
examples_path = output_dir / 'pubmed_fallback_rerank_examples.csv'
if examples_path.exists():
    examples = pd.read_csv(examples_path)
    print(examples['case_group'].value_counts(dropna=False))
    cols = [
        'case_group', 'metabolite_name', 'gold_refmet_name', 'gold_refmet_cids',
        'embedding_original_order_same_cid_rank', 'pubmed_tuned_rerank_same_cid_rank',
        'pubmed_tuned_rerank_top1_name', 'pubmed_tuned_rerank_top1_cids',
        'pubmed_tuned_rerank_top2_name', 'pubmed_tuned_rerank_top2_cids',
    ]
    display(examples[[c for c in cols if c in examples.columns]].head(30))
else:
    print('Examples file not found yet:', examples_path)

## 7. Plot Summary

In [ ]:
from IPython.display import Image, display

plots_dir = output_dir / 'plots'
for name in ['pubmed_fallback_MRR.png', 'pubmed_fallback_Hitat1.png', 'pubmed_fallback_Hitat10.png', 'pubmed_fallback_Hitat256.png']:
    path = plots_dir / name
    if path.exists():
        display(Image(filename=str(path)))